In [3]:
### 중복된 화합물이 많아서 다시 데이터 정리 진행
#이전 코드를 다시 살펴보니, skin_irritation_cleaned.csv를 만들 때 분명히 중복 제거를 했음.
#그런데도 여전히 동일 구조가 있는 이유???

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

df = pd.read_csv('skin_irritation_cleaned.csv')
df.shape

df.columns

df['SMILES'].str.contains('.', regex=False)

'''
skin_irritation_cleaned.csv를 준비할 때, 분명히 중복 제거를 했지만 여전히 중복된 화합물이 있는 이유: 화합물 이름을 기준으로 중복 제거를 했기 때문.
아래 테이블을 보면 index 4번과 54번이 동일한 구조를 갖고 있음. 그런데 이름을 보면 4번은 sulphate, 54번은 sulfate로 적혀있음.
화합물 이름으로만 중복 제거를 하면 놓칠 수 있음. 그래서 중복 제거를 할 때는 여러 정보를 기준으로 중복 체크를 진행해야 함.
'''
df.loc[df['SMILES'].str.contains('.', regex=False), ['Chemical_Name','SMILES']]

#https://docs.chemaxon.com/latest/formats_chemaxon-extended-smiles-and-smarts-cxsmiles-and-cxsmarts.html
#아래 선택된 smiles 코드에 대한 설명 (Notebook LM에 링크 넣고 확인 ㄱㄱ)
df.loc[df['SMILES'].str.contains('|', regex=False), ['Chemical_Name','SMILES']]

Chem.MolFromSmiles('[H]OCCO |lp:1:2,4:2,Sg:n:1,2,3::ht|')

Chem.MolFromSmiles('C*.C*.OCCOCCO |lp:4:2,7:2,10:2,m:1:5.6,3:8.9|')

# | 기호가 있는 화합물은 구조가 명확하지 않아서 제거.
df_clean_smi = df.loc[~df['SMILES'].str.contains('|', regex=False)]

df_clean_smi.loc[df_clean_smi['SMILES'].str.contains('.', regex=False), ['Chemical_Name','SMILES','label']]

from rdkit.Chem.SaltRemover import SaltRemover

# Initialize with default RDKit salt definitions
remover = SaltRemover()

# Example: Ethanol with a Sodium ion
for index, row in df_clean_smi.iterrows():
    smi = row['SMILES']
    mol = Chem.MolFromSmiles(smi)
    
    if '.' in smi:
        stripped_mol = remover.StripMol(mol) #salt 제거
        stripped_smi = Chem.MolToSmiles(stripped_mol)
        smi_list = stripped_smi.split('.')

        #salt를 제거한 후에도, 분자 구조가 여러개 들어 있음. 가장 큰 분자를 active ingredient로 생각하고 추출. 
        smi_active = ''
        for each_smi in smi_list:
            if len(each_smi) > len(smi_active): #새로운 분자의 smiles 코드가 더 길다면, 더 큰 구조로 판단.
                smi_active = each_smi
        smi_without_salt = smi_active
        
        print (index, smi_without_salt)
    else:
        smi_without_salt = smi

    df_clean_smi.loc[index, 'smi_clean']=smi_without_salt
#4번 인덱스는 smiles 코드가 아무것도 없음. [OH][Na]여서 염제거 과정에서 다 삭제된 것으로 보임.

df_clean_smi

#smiles 표준화
for index, row in df_clean_smi.iterrows():
    smi = row['smi_clean']
    mol = Chem.MolFromSmiles(smi)
    smi_standard = Chem.MolToSmiles(mol)
    df_clean_smi.loc[index, 'standardized_smi'] = smi_standard

df_clean_smi[['Chemical_Name','standardized_smi','label']]

unique_smi = set(df_clean_smi['standardized_smi'].to_list())
len(unique_smi)

unique_smi.remove('') #smiles 코드 중 ''이 보여서 제거.

len(unique_smi)

for smi in unique_smi:
    print (smi)

df_clean_smi.groupby('standardized_smi')['label'].nunique()

df_filtered = df_clean_smi.groupby('standardized_smi').filter(lambda x: x['label'].nunique() == 1)

df_filtered.groupby('standardized_smi')['label'].nunique()


df_filtered.groupby('standardized_smi')['label'].nunique().shape

df_filtered.shape

df_unique = df_filtered.drop_duplicates(subset='standardized_smi')

df_unique.shape

df_unique['label'].value_counts()

desc_total = []
for index, row in df_unique.iterrows():
    smi = row['SMILES']
    mol = Chem.MolFromSmiles(smi)
    desc = Descriptors.CalcMolDescriptors(mol)
    desc_total.append(desc)

df_desc = pd.DataFrame(desc_total)

df_desc

df_id = df_unique[['Chemical_Name','standardized_smi','label']]

df_id #중간에 subset 하면서 index값이 1 다음 4로 건너뛰는 것을 확인.

df_final = pd.concat([df_id,df_desc],axis=1)


df_final #concatenation 실패. row가 40개가 나와야 하는데 51개. (df_id와 df_desc의 index 번호가 일치하지 않기 때문)


df_id.reset_index()

df_id.reset_index(drop=True)

df_final = pd.concat([df_id.reset_index(drop=True),df_desc],axis=1)

df_final #concatenation 성공 (row 개수 40개)


4 
11 CCCCCCCCCCCCOS(=O)(=O)[O-]
12 O=C([O-])[O-]
24 O=C([O-])[O-]
48 O=C([O-])CN(CCN(CC(=O)[O-])CC(=O)O)CC(=O)O
57 CCCCCCCCCCCCOS(=O)(=O)[O-]
58 O=C([O-])[O-]
64 O=C([O-])[O-]
79 O=C([O-])CN(CCN(CC(=O)[O-])CC(=O)O)CC(=O)O
O=C(OCc1ccccc1)c1ccccc1O
CSc1ccc(C=O)cc1
CC(C)=CCCC(C)CCO
CCCCCCCCCCO
CCCCCCCCC(=O)O
CC(O)COC(C)(C)C
CCCCCCCCCCCCCCCC(=O)O
CCCCCCCCCCCCCC(=O)O
COCC(COC)(CC(C)C)C(C)C
OCCN(CCO)CCO
CC(C)O
C=CCCC(=O)C1=CCCC2(CCCC2)C1
CCCCCCCCCCCCCCCC(=O)OC
CCCCCCCOC(=O)CCC
NC(CO)(CO)CO
ClCCCCBr
CCCCCCCCCCCCCCCC(=O)OC(C)C
CC1=CCC(C(C)(C)O)CC1
CCCCCCCC(=O)O
CC(O)CO
CCCCCC(=O)OC
CCCCCCOC(=O)c1ccccc1O
CCCCCCCCCCCCCC(=O)OC(C)C
CC(=O)OC(C)(C)C1CC=C(C)CC1
CS(C)=O
CCCCCCBr
CC(O)C(=O)O
Cc1c[nH]nc1C
CCCSSCCC
CCCCCCC(=O)O
CCCCCCCCCC(=O)O
CCCCCCCCCCCCO
CCCCCCCCCCCCOS(=O)(=O)[O-]
CCCCCCCCCCCCCCO
CCCCCCCCCCCC(=O)OC
CC(C)=CCCC(C)=CCO
O=C(O)Cc1cccc2ccccc12
CCCCOC(=O)c1ccccc1
CCCCCCO
CCCCCCCCO
CCCCCCC=O
O=C([O-])[O-]
COC(=O)c1ccccc1N=CCC(C)CCCC(C)(C)O
CC(C)c1ccccc1C(C)c1ccccc1
CCCCCO
C=C(C)C(=O)OCCCC
O=

,Chemical_Name,standardized_smi,label,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,Heptanal,CCCCCCC=O,1,9.768009,9.768009,0.750000,0.750000,0.395123,9.125000,114.188,...,0,0,0,0,0,0,0,0,4,0
1,Sodium hydroxide,,1,0.000000,0.000000,0.000000,0.000000,0.262038,0.000000,39.997,...,0,0,0,0,0,0,0,0,0,0
2,4-Methylthio benzaldehyde,CSc1ccc(C=O)cc1,0,10.199308,10.199308,0.734074,0.734074,0.477177,9.300000,152.218,...,1,0,0,0,0,0,0,0,0,0
3,Heptyl butyrate,CCCCCCCOC(=O)CCC,0,10.909695,10.909695,0.043526,-0.043526,0.429221,10.000000,186.295,...,0,0,0,0,0,0,0,0,4,0
4,Hydroxycitronellal,COC(=O)c1ccccc1N=CCC(C)CCCC(C)(C)O,0,11.646816,11.646816,0.369467,-0.590016,0.579721,13.318182,305.418,...,0,0,0,0,0,0,0,0,0,0
5,Methyl caproate,CCCCCC(=O)OC,0,10.464402,10.464402,0.094028,-0.094028,0.427720,9.111111,130.187,...,0,0,0,0,0,0,0,0,2,0
6,1-(2-Isopropylphenyl)-1-phenyle- thane (Mixtur...,CC(C)c1ccccc1C(C)c1ccccc1,0,2.288241,2.288241,0.467407,0.467407,0.690772,12.705882,224.347,...,0,0,0,0,0,0,0,0,0,0
7,Benzyl salicylate,O=C(OCc1ccccc1)c1ccccc1O,0,11.656605,11.656605,0.064295,-0.521340,0.821476,9.882353,228.247,...,0,0,0,0,0,0,0,0,0,0
8,Sodium dodecyl sulphate,CCCCCCCCCCCCOS(=O)(=O)[O-],1,10.113810,10.113810,0.000000,-4.484382,0.225415,11.222222,288.385,...,0,0,0,0,0,0,0,0,9,0
9,1-Bromo-4-chlorobutane,ClCCCCBr,0,5.359460,5.359460,0.797222,0.797222,0.451849,9.000000,171.465,...,0,0,0,0,0,0,0,0,1,0


In [5]:
#https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
#logistic regression 모델 만들어보기

from sklearn.linear_model import LogisticRegression

y = df_final['label']
x = df_final.drop(columns=['Chemical_Name','standardized_smi','label'])

y

x

clf = LogisticRegression()


In [6]:
# NaN이 있는 컬럼 확인
nan_columns = x.columns[x.isna().any()]

print("NaN 포함된 컬럼 개수:", len(nan_columns))
print(nan_columns)

NaN 포함된 컬럼 개수: 8
Index(['BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BCUT2D_CHGHI', 'BCUT2D_CHGLO',
       'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW'],
      dtype='str')


In [7]:
# NaN 있는 컬럼 제거
x_clean = x.drop(columns=nan_columns)

# 확인
print("원래 shape:", x.shape)
print("정리 후 shape:", x_clean.shape)

원래 shape: (40, 217)
정리 후 shape: (40, 209)


In [8]:
print(x_clean.isna().sum().sum())  # 0이면 깨끗한 상태

0


In [9]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)  # 반복수 늘려주는게 안전함
clf.fit(x_clean, y)

print("모델 학습 완료!")

모델 학습 완료!


C:\Users\DS\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
accuracy = clf.score(x_clean, y)

print("모델 정확도:", accuracy)

모델 정확도: 1.0


In [11]:
from rdkit import Chem
from rdkit.Chem import MACCSkeys
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator, GetRDKitFPGenerator
from rdkit.DataStructs import ConvertToNumpyArray

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

In [12]:
df_model = df_unique[['Chemical_Name', 'standardized_smi', 'label']].copy()

# 빈 문자열 제거
df_model = df_model[df_model['standardized_smi'] != ''].copy()

# Mol 객체 생성
df_model['Mol'] = df_model['standardized_smi'].apply(Chem.MolFromSmiles)

# 혹시 Mol 생성 실패한 행 제거
df_model = df_model[df_model['Mol'].notnull()].copy()

df_model.shape

(39, 4)

In [19]:
# descriptor dataframe 다시 정리
desc_total = []

for idx, row in df_model.iterrows():
    mol = row['Mol']
    desc = Descriptors.CalcMolDescriptors(mol)
    desc_total.append(desc)

df_desc = pd.DataFrame(desc_total)

# NaN 포함 컬럼 제거
nan_cols = df_desc.columns[df_desc.isna().any()]
df_desc_clean = df_desc.drop(columns=nan_cols)

# y 준비
y_desc = df_model['label'].reset_index(drop=True)

# x 준비
x_desc = df_desc_clean.reset_index(drop=True)

print("Descriptor shape:", x_desc.shape)

Descriptor shape: (39, 217)


In [22]:
print(x_rdkitfp.shape)
print(len(y))

(39, 2048)
40


In [23]:
from rdkit import Chem
from rdkit.Chem.rdFingerprintGenerator import GetRDKitFPGenerator
from rdkit.DataStructs import ConvertToNumpyArray
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# 1. RDKitFP용 데이터 다시 준비
df_rdkit = df_unique[['standardized_smi', 'label']].copy()

# 빈 문자열 제거
df_rdkit = df_rdkit[df_rdkit['standardized_smi'] != ''].copy()

# Mol 생성
df_rdkit['Mol'] = df_rdkit['standardized_smi'].apply(Chem.MolFromSmiles)

# Mol 생성 실패한 행 제거
df_rdkit = df_rdkit[df_rdkit['Mol'].notnull()].reset_index(drop=True)

print(df_rdkit.shape)   # 여기서 39행인지 확인

# 2. RDKit fingerprint 만들기
rdkit_gen = GetRDKitFPGenerator(fpSize=2048)

rdkitfp_list = []
for mol in df_rdkit['Mol']:
    fp = rdkit_gen.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=int)
    ConvertToNumpyArray(fp, arr)
    rdkitfp_list.append(arr)

x_rdkitfp = pd.DataFrame(rdkitfp_list)

# 3. y도 같은 df_rdkit 기준으로 맞추기
y_rdkit = df_rdkit['label']

print(x_rdkitfp.shape)   # (39, 2048)
print(len(y_rdkit))      # 39

# 4. 모델 학습
clf = LogisticRegression(max_iter=1000)
clf.fit(x_rdkitfp, y_rdkit)

# 5. 정확도 확인
print("RDKitFP accuracy:", clf.score(x_rdkitfp, y_rdkit))

(39, 3)
(39, 2048)
39
RDKitFP accuracy: 0.9230769230769231


In [1]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem import rdFingerprintGenerator

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# -----------------------------
# 1. 데이터 불러오기 및 정리
# -----------------------------
df = pd.read_csv('skin_irritation_cleaned.csv')

# | 포함된 구조 제거
df_clean_smi = df.loc[~df['SMILES'].str.contains('|', regex=False)].copy()

# salt 제거 준비
remover = SaltRemover()

# salt 제거 + active ingredient 추출
for index, row in df_clean_smi.iterrows():
    smi = row['SMILES']
    mol = Chem.MolFromSmiles(smi)

    if mol is None:
        df_clean_smi.loc[index, 'smi_clean'] = np.nan
        continue

    if '.' in smi:
        stripped_mol = remover.StripMol(mol)
        if stripped_mol is None:
            df_clean_smi.loc[index, 'smi_clean'] = np.nan
            continue

        stripped_smi = Chem.MolToSmiles(stripped_mol)
        smi_list = stripped_smi.split('.')

        # 가장 긴 SMILES를 active ingredient로 선택
        smi_active = ''
        for each_smi in smi_list:
            if len(each_smi) > len(smi_active):
                smi_active = each_smi

        smi_without_salt = smi_active
    else:
        smi_without_salt = smi

    df_clean_smi.loc[index, 'smi_clean'] = smi_without_salt

# 빈 값 제거
df_clean_smi = df_clean_smi.dropna(subset=['smi_clean']).copy()
df_clean_smi = df_clean_smi[df_clean_smi['smi_clean'] != ''].copy()

# SMILES 표준화
for index, row in df_clean_smi.iterrows():
    smi = row['smi_clean']
    mol = Chem.MolFromSmiles(smi)

    if mol is None:
        df_clean_smi.loc[index, 'standardized_smi'] = np.nan
    else:
        smi_standard = Chem.MolToSmiles(mol)
        df_clean_smi.loc[index, 'standardized_smi'] = smi_standard

df_clean_smi = df_clean_smi.dropna(subset=['standardized_smi']).copy()
df_clean_smi = df_clean_smi[df_clean_smi['standardized_smi'] != ''].copy()

# 같은 standardized_smi인데 label이 여러 개인 경우 제거
df_filtered = df_clean_smi.groupby('standardized_smi').filter(
    lambda x: x['label'].nunique() == 1
).copy()

# 중복 구조 제거
df_unique = df_filtered.drop_duplicates(subset='standardized_smi').copy()
df_unique = df_unique.reset_index(drop=True)

print("최종 사용 화합물 개수:", df_unique.shape[0])
print(df_unique['label'].value_counts())

# -----------------------------
# 2. descriptor 계산
# -----------------------------
desc_total = []

for index, row in df_unique.iterrows():
    smi = row['standardized_smi']   # 여기서는 cleaned/standardized SMILES 사용
    mol = Chem.MolFromSmiles(smi)

    if mol is None:
        desc_total.append({})
    else:
        desc = Descriptors.CalcMolDescriptors(mol)
        desc_total.append(desc)

df_desc = pd.DataFrame(desc_total)

# 무한대/문자 등 예외처리
df_desc = df_desc.replace([np.inf, -np.inf], np.nan)

print("Descriptor 개수:", df_desc.shape[1])

# -----------------------------
# 3. fingerprint 계산
# -----------------------------
# Morgan fingerprint (radius=2, 2048 bits)
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

fp_list = []
for smi in df_unique['standardized_smi']:
    mol = Chem.MolFromSmiles(smi)
    fp = fp_gen.GetFingerprint(mol)
    fp_array = list(fp)
    fp_list.append(fp_array)

df_fp = pd.DataFrame(fp_list, columns=[f'FP_{i}' for i in range(2048)])

print("Fingerprint bit 개수:", df_fp.shape[1])

# -----------------------------
# 4. label / CV 설정
# -----------------------------
y = df_unique['label'].copy()

# label이 문자열이면 숫자로 바꾸기
if y.dtype == 'O':
    y = y.astype('category').cat.codes

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# descriptor용 pipeline
desc_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=5000, random_state=42))
])

# fingerprint용 pipeline
fp_model = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('clf', LogisticRegression(max_iter=5000, random_state=42))
])

scoring = {
    'accuracy': 'accuracy',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

# -----------------------------
# 5. descriptor 하나씩 평가
# -----------------------------
desc_results = []

for col in df_desc.columns:
    X_one = df_desc[[col]]

    try:
        scores = cross_validate(
            desc_model,
            X_one,
            y,
            cv=cv,
            scoring=scoring,
            error_score='raise'
        )

        desc_results.append({
            'feature_name': col,
            'feature_type': 'descriptor',
            'mean_accuracy': scores['test_accuracy'].mean(),
            'std_accuracy': scores['test_accuracy'].std(),
            'mean_f1': scores['test_f1'].mean(),
            'std_f1': scores['test_f1'].std(),
            'mean_roc_auc': scores['test_roc_auc'].mean(),
            'std_roc_auc': scores['test_roc_auc'].std()
        })
    except Exception as e:
        desc_results.append({
            'feature_name': col,
            'feature_type': 'descriptor',
            'mean_accuracy': np.nan,
            'std_accuracy': np.nan,
            'mean_f1': np.nan,
            'std_f1': np.nan,
            'mean_roc_auc': np.nan,
            'std_roc_auc': np.nan
        })

df_desc_results = pd.DataFrame(desc_results)
df_desc_results = df_desc_results.sort_values(
    by='mean_roc_auc', ascending=False
).reset_index(drop=True)

print("\n=== Descriptor 단일 변수 성능 상위 10개 ===")
print(df_desc_results.head(10))

best_desc = df_desc_results.iloc[0]
print("\n=== 최고 성능 Descriptor ===")
print(best_desc)

# -----------------------------
# 6. fingerprint bit 하나씩 평가
# -----------------------------
fp_results = []

for col in df_fp.columns:
    X_one = df_fp[[col]]

    # 값이 전부 같으면 학습 의미 없음
    if X_one[col].nunique() < 2:
        fp_results.append({
            'feature_name': col,
            'feature_type': 'fingerprint',
            'mean_accuracy': np.nan,
            'std_accuracy': np.nan,
            'mean_f1': np.nan,
            'std_f1': np.nan,
            'mean_roc_auc': np.nan,
            'std_roc_auc': np.nan
        })
        continue

    try:
        scores = cross_validate(
            fp_model,
            X_one,
            y,
            cv=cv,
            scoring=scoring,
            error_score='raise'
        )

        fp_results.append({
            'feature_name': col,
            'feature_type': 'fingerprint',
            'mean_accuracy': scores['test_accuracy'].mean(),
            'std_accuracy': scores['test_accuracy'].std(),
            'mean_f1': scores['test_f1'].mean(),
            'std_f1': scores['test_f1'].std(),
            'mean_roc_auc': scores['test_roc_auc'].mean(),
            'std_roc_auc': scores['test_roc_auc'].std()
        })
    except Exception as e:
        fp_results.append({
            'feature_name': col,
            'feature_type': 'fingerprint',
            'mean_accuracy': np.nan,
            'std_accuracy': np.nan,
            'mean_f1': np.nan,
            'std_f1': np.nan,
            'mean_roc_auc': np.nan,
            'std_roc_auc': np.nan
        })

df_fp_results = pd.DataFrame(fp_results)
df_fp_results = df_fp_results.sort_values(
    by='mean_roc_auc', ascending=False
).reset_index(drop=True)

print("\n=== Fingerprint bit 단일 변수 성능 상위 10개 ===")
print(df_fp_results.head(10))

best_fp = df_fp_results.iloc[0]
print("\n=== 최고 성능 Fingerprint bit ===")
print(best_fp)

# -----------------------------
# 7. descriptor vs fingerprint 최종 비교
# -----------------------------
best_overall = pd.concat([
    df_desc_results.head(1),
    df_fp_results.head(1)
], axis=0).sort_values(by='mean_roc_auc', ascending=False).reset_index(drop=True)

print("\n=== 최종 1등 feature ===")
print(best_overall.iloc[0])

# -----------------------------
# 8. 결과 저장
# -----------------------------
df_desc_results.to_csv('single_descriptor_results.csv', index=False)
df_fp_results.to_csv('single_fingerprint_results.csv', index=False)

print("\n결과 저장 완료:")
print("- single_descriptor_results.csv")
print("- single_fingerprint_results.csv")

최종 사용 화합물 개수: 39
label
0    26
1    13
Name: count, dtype: int64
Descriptor 개수: 217
Fingerprint bit 개수: 2048

=== Descriptor 단일 변수 성능 상위 10개 ===
        feature_name feature_type  mean_accuracy  std_accuracy   mean_f1  \
0                Ipc   descriptor       0.667857      0.053690  0.000000   
1     MaxEStateIndex   descriptor       0.692857      0.056919  0.260000   
2  MaxAbsEStateIndex   descriptor       0.692857      0.056919  0.260000   
3             AvgIpc   descriptor       0.692857      0.056919  0.200000   
4       BCUT2D_CHGHI   descriptor       0.589286      0.093131  0.000000   
5              Chi2n   descriptor       0.667857      0.095565  0.200000   
6        EState_VSA2   descriptor       0.592857      0.114286  0.057143   
7      BCUT2D_LOGPHI   descriptor       0.664286      0.072668  0.180000   
8     HeavyAtomMolWt   descriptor       0.746429      0.174379  0.360000   
9       BCUT2D_CHGLO   descriptor       0.639286      0.098716  0.100000   

     std_f1  mean_

In [3]:
print("df shape:", df.shape)
print("df_unique shape:", df_unique.shape)
print("df_desc shape:", df_desc.shape)

print("df label 개수:", df['label'].shape)
print("df_unique label 개수:", df_unique['label'].shape)

df shape: (81, 27)
df_unique shape: (39, 29)
df_desc shape: (39, 217)
df label 개수: (81,)
df_unique label 개수: (39,)


In [5]:
model = LogisticRegression(max_iter=5000)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = df_desc
y = df_unique['label']

print(X.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("학습 완료")
print("예측값:", pred)

(39, 217)
(39,)
학습 완료
예측값: [1 1 0 0 0 1 1 1]


C:\Users\DS\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
